# Sieci neuronowe i deep learning - Lab 6

## Modele sekwencyjne i attention

PyTorch.

W tym laboratorium pracujemy na plikach `winnie.txt` i `peterpan.txt`. Budujemy znakowy model jezykowy: z fragmentu tekstu model przewiduje kolejny znak. Na tym prostym zadaniu porownujemy RNN, LSTM/GRU oraz prosty mechanizm attention.

## Zadanie 1

Przygotuj dane tekstowe:

- wczytaj `winnie.txt` i `peterpan.txt`,
- wybierz korpus do treningu albo polacz oba teksty,
- opcjonalnie podmien teksty na wlasny korpus, jesli zachowasz dalszy format `raw_text`,
- oczysc tekst w prosty, jawny sposob,
- zbuduj slownik znakow,
- przygotuj pary: sekwencja znakow -> nastepny znak.


In [ ]:
from pathlib import Path
import re
import textwrap

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

SEQ_LENGTH = 80
STEP = 3
MAX_CHARS = 80000
MAX_EXAMPLES = 14000
BATCH_SIZE = 128
EMBEDDING_DIM = 48
HIDDEN_DIM = 96
TRAIN_EPOCHS_SHORT = 3
TRAIN_EPOCHS_GENERATIVE = 5

# Propozycja wariantu GPU, jesli grupa chce lepsze probki tekstu:
# MAX_CHARS = 160000
# MAX_EXAMPLES = 30000
# TRAIN_EPOCHS_GENERATIVE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


def find_data_dir():
    candidates = [Path("../dane"), Path("laboratoria/dane"), Path("dane"), Path("/content/dane")]
    for candidate in candidates:
        if (candidate / "winnie.txt").exists() and (candidate / "peterpan.txt").exists():
            return candidate

    fallback = Path("dane")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def ensure_text_files(data_dir):
    sources = {
        "peterpan.txt": "https://www.gutenberg.org/cache/epub/16/pg16.txt",
        "winnie.txt": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    }
    for filename, url in sources.items():
        path = data_dir / filename
        if path.exists():
            continue
        try:
            import urllib.request
            print(f"Pobieram fallback tekstowy: {filename}")
            urllib.request.urlretrieve(url, path)
        except Exception as exc:
            sample = (
                "This is a short fallback text for sequence modelling. "
                "It is used only when the notebook cannot download the public-domain corpus. "
            )
            path.write_text(sample * 2000, encoding="utf-8")
            print(f"Nie udalo sie pobrac {filename}; zapisano krotki fallback. Przyczyna: {exc}")


DATA_DIR = find_data_dir()
ensure_text_files(DATA_DIR)
print("Dane:", DATA_DIR.resolve())
print("Device:", DEVICE)


In [ ]:
# Zadanie 1a: wczytanie i czyszczenie tekstu.


def read_text(name):
    # TODO: wczytaj plik z DATA_DIR.
    raise NotImplementedError


def clean_text(text):
    # TODO: zamien tekst na male litery i uporzadkuj biale znaki.
    raise NotImplementedError


texts = {
    "winnie": ...,
    "peterpan": ...,
}

# TODO: wybierz jeden korpus albo polacz oba teksty.
CORPUS_NAME = ...
raw_text = ...

print(...)
print(raw_text[:500])

In [ ]:
# Zadanie 1b: slownik znakow i kodowanie.

# TODO:
# - zbuduj posortowana liste znakow,
# - przygotuj mapowania char_to_id oraz id_to_char,
# - zakoduj raw_text jako tablice liczb.

chars = ...
char_to_id = ...
id_to_char = ...
vocab_size = ...
encoded = ...

print(...)

In [ ]:
# Zadanie 1c: pary sekwencja -> nastepny znak.


def make_examples(encoded_text, seq_length=SEQ_LENGTH, step=STEP, max_examples=MAX_EXAMPLES):
    # TODO: przygotuj macierze X oraz y.
    # X[i] ma zawierac seq_length kolejnych znakow, y[i] kolejny znak po tej sekwencji.
    raise NotImplementedError


X, y = ...

# TODO: wydziel train/validation bez mieszania kolejnosci tekstu.
X_train, y_train = ...
X_val, y_val = ...

# TODO: zbuduj TensorDataset i DataLoader dla treningu i walidacji.
train_dataset = ...
val_dataset = ...
train_loader = ...
val_loader = ...

print(...)
print(...)

In [ ]:
# Funkcje pomocnicze.


def plot_history(history, title):
    # TODO: narysuj loss i accuracy dla train/validation.
    raise NotImplementedError


def perplexity(loss):
    # TODO: przelicz cross-entropy loss na perplexity.
    raise NotImplementedError


def print_score_table(rows):
    # TODO: wypisz porownanie modeli.
    raise NotImplementedError


def count_parameters(model):
    # TODO: policz tylko trenowalne parametry.
    raise NotImplementedError


def run_epoch(model, loader, loss_fn, optimizer=None):
    # TODO: jedna epoka treningowa albo walidacyjna.
    raise NotImplementedError


def train_torch_model(model, name, epochs=3, learning_rate=0.001):
    # TODO: petla treningowa, historia, wiersz wynikow.
    raise NotImplementedError

## Zadanie 2

Wytrenuj bazowy model rekurencyjny.

Porownaj, jak zachowuje sie prosty RNN na zadaniu predykcji nastepnego znaku. Zapisz loss, validation loss, accuracy i perplexity.

In [ ]:
# Zadanie 2: bazowy model rekurencyjny.


class RecurrentLM(nn.Module):
    def __init__(self, kind="rnn", vocab_size=vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # TODO:
        # - Embedding,
        # - RNN/LSTM/GRU zaleznie od kind,
        # - Dropout,
        # - Linear(vocab_size).
        raise NotImplementedError

    def forward(self, x):
        # TODO: zwroc logits dla nastepnego znaku.
        raise NotImplementedError


torch_rows = []

rnn_model, rnn_history, rnn_row = ...
torch_rows.append(rnn_row)

plot_history(...)
print_score_table(torch_rows)

## Zadanie 3

Porownaj LSTM i GRU.

Sprawdz, czy bramkowane modele rekurencyjne latwiej ucza sie zaleznosci w tekscie niz prosty RNN. Wygeneruj probki tekstu dla kilku temperatur i ustawien top-k. Nie oczekujemy jeszcze pieknej prozy: sprawdzamy, czy model uczy sie lokalnej skladni znakow, spacji i krotkich slow lepiej niz losowy generator.


In [ ]:
# Zadanie 3a: LSTM i GRU.

# TODO: wytrenuj LSTM oraz GRU, dopisz wyniki do tabeli i narysuj historie.

lstm_model, lstm_history, lstm_row = ...
torch_rows.append(lstm_row)

gru_model, gru_history, gru_row = ...
torch_rows.append(gru_row)

plot_history(...)
plot_history(...)
print_score_table(torch_rows)

In [ ]:
# Zadanie 3b: generowanie tekstu.


def sample_from_logits(logits, temperature=1.0, top_k=None):
    # TODO:
    # - podziel logits przez temperature,
    # - opcjonalnie zostaw tylko top_k najbardziej prawdopodobnych znakow,
    # - policz softmax,
    # - wylosuj indeks znaku z otrzymanego rozkladu.
    raise NotImplementedError


def encode_seed(seed):
    # TODO: zakoduj seed i dopasuj dlugosc do SEQ_LENGTH.
    raise NotImplementedError


def make_generation_seed(example_index=0, length=60):
    # TODO: wez koncowke wybranej sekwencji walidacyjnej jako realistyczny prompt.
    raise NotImplementedError


def generate_text(model, seed, num_chars=300, temperature=0.7, top_k=10):
    # TODO: generuj tekst autoregresyjnie znak po znaku.
    raise NotImplementedError


seed = ...
settings = [(0.45, 5), (0.7, 10), (0.9, 20)]

for temperature, top_k in settings:
    print(...)
    print(...)


## Zadanie 4

Dodaj attention do modelu sekwencyjnego.

W poprzednich zadaniach model RNN/LSTM/GRU podejmowal decyzje na podstawie ostatniego stanu ukrytego. Teraz sprawdzamy inny wariant: GRU zwraca wszystkie stany ukryte, a proste attention pooling uczy sie, ktore pozycje sekwencji sa najwazniejsze dla predykcji nastepnego znaku.

Celem nie jest zalozenie, ze attention na pewno poprawi wynik. Celem jest uczciwe porownanie: trenujemy model z attention na tym samym zadaniu, dodajemy go do tabeli i sprawdzamy validation loss, perplexity oraz accuracy. Na koncu generujemy tez krotka probke tekstu z modelu `GRU+attention`, zeby porownanie nie konczylo sie sama tabela.


In [ ]:
# Zadanie 4a: GRU z attention pooling.


class AttentionGRULM(nn.Module):
    def __init__(self, vocab_size=vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # TODO:
        # - Embedding,
        # - GRU zwracajacy cala sekwencje stanow,
        # - warstwa score dla kazdej pozycji,
        # - Dropout,
        # - Linear(vocab_size).
        raise NotImplementedError

    def forward(self, x, return_attention=False):
        # TODO:
        # - policz wagi attention po osi czasu,
        # - zbuduj wektor kontekstu jako wazona sume,
        # - opcjonalnie zwroc wagi attention.
        raise NotImplementedError


attention_model, attention_history, attention_row = ...
torch_rows.append(attention_row)

plot_history(...)
print_score_table(torch_rows)


In [ ]:
# Zadanie 4b: wizualizacja i interpretacja wag attention.

# TODO:
# - zbuduj model albo wywolanie zwracajace wagi attention,
# - wybierz probke walidacyjna,
# - policz ranking prawdziwego nastepnego znaku,
# - narysuj wagi attention po pozycjach,
# - wypisz znaki z najwyzszym prawdopodobienstwem oraz pozycje z najwyzsza waga.

sample_index = ...
sample_probabilities = ...
sample_weights = ...

# TODO: wygeneruj probke tekstu z attention_model dla tych samych zasad probkowania:
#       seed z walidacji, konserwatywne temperatury i top_k.
# TODO: porownaj jakosciowo tekst z GRU i GRU+attention.

...
